In [1]:
import torch

# your journey starts with one step
inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your     (x^1)
    [0.55, 0.87, 0.66], # journey  (x^2)
    [0.57, 0.85, 0.64], # starts   (x^3)
    [0.22, 0.58, 0.33], # with     (x^4)
    [0.77, 0.25, 0.10], # one      (x^5)
    [0.05, 0.80, 0.55]  # step     (x^6)
] )

# For all row

In [2]:
d_in = inputs.shape[1]    # (6, 3) -> 3     
d_out = 2                 # embedding_size = 2

In [3]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in,d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in,d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in,d_out), requires_grad=False)

In [4]:
keys = inputs @ W_key         # (6,3) @ (3,2) -> (6,2)
queries = inputs @ W_query
values = inputs @ W_value

In [16]:
attention_score = queries @ keys.T       #(6,2) @ (2,6) -> (6,6)
attention_score

tensor([[0.9231, 1.3545, 1.3241, 0.7910, 0.4032, 1.1330],
        [1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
        [1.2544, 1.8284, 1.7877, 1.0654, 0.5508, 1.5238],
        [0.6973, 1.0167, 0.9941, 0.5925, 0.3061, 0.8475],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707, 0.7307],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]])

In [17]:
d_k = keys.shape[-1]         #(6,2) -> 2
attention_weight = torch.softmax((attention_score / d_k**0.5), dim=-1)
attention_weight

tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])

In [18]:
context_vector = attention_weight @ values      #(6,6) @ (6,2) -> (6,2)
context_vector

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]])

# For 2nd row [journey]

In [5]:
input_2  = inputs[1]      # (1,3)
input_2

tensor([0.5500, 0.8700, 0.6600])

In [6]:
query_2 = input_2 @ W_query     # (1,3) @ (3,2) -> (1,2)
key_2 = input_2 @ W_key
value_2 = input_2 @ W_value

##### (w22) =  For 'Journey' --> key_2  

In [7]:
attention_score_22 = query_2.dot(key_2)
attention_score_22

tensor(1.8524)

##### (w21, w22, w23, ..., w2T) =  For 'Journey' --> keys 

In [8]:
attention_score_2 = query_2 @ keys.T
attention_score_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])

In [12]:
d_k = keys.shape[-1]   #(6,2) -> 2

attention_weight_2 = torch.softmax(attention_score_2 / d_k**0.5, dim=-1)
attention_weight_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])

In [15]:
context_vector_2 = attention_weight_2 @ values    # (1,6) @ (6,2) -> (1,2)
context_vector_2

tensor([0.3061, 0.8210])

# Implement into class (nn.Parameter)

In [19]:

import torch.nn as nn

class SelfAttentionOwnClass(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in,d_out))
        self.W_key = nn.Parameter(torch.rand(d_in,d_out))
        self.W_value = nn.Parameter(torch.rand(d_in,d_out))
        
    def forward(self,x):
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value
        
        attention_score = queries @ keys.T
        d_k = keys.shape[-1]
        attention_weight = torch.softmax(
            attention_score/d_k**0.5, dim=-1
        )
        context_vector = attention_weight @ values
        return context_vector

In [20]:
d_in = inputs.shape[1]    # (6, 3) -> 3     
d_out = 2                 # embedding_size = 2

In [21]:
torch.manual_seed(123)
selfAttention = SelfAttentionOwnClass(d_in,d_out)

In [22]:
selfAttention(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

# Implement into class (nn.Linear)

In [26]:

class SelfAttentionOwnClassUpdated(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
    def forward(self,x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        
        attention_score = queries @ keys.T
        
        d_k = keys.shape[-1]
        attention_weight = torch.softmax(
            attention_score/d_k**0.5, dim=-1
        )
        context_vector = attention_weight @ values
        return context_vector

In [29]:
torch.manual_seed(789)
selfAttention2 = SelfAttentionOwnClassUpdated(d_in, d_out)

In [30]:
selfAttention2(inputs)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)